# Vérifier les références SQL avec pandas

La cellule compare les résultats SQL enregistrés dans `reference-td1.csv` à un calcul indépendant effectué avec pandas à partir de `data/orders.csv`, `data/product.csv` et `data/customers.csv`.

In [ ]:
import pandas as pd
from pathlib import Path

root = Path.cwd()
data_path = root / "data"
reference_path = root / "reference-td1.csv"

# This CSV contains the values previously calculated by running the SQL queries.
sql_reference = pd.read_csv(reference_path)
products = pd.read_csv(data_path / "product.csv")
customers = pd.read_csv(data_path / "customers.csv", parse_dates=["birth_date"])
orders = pd.read_csv(data_path / "orders.csv", parse_dates=["date"])

# Join CSV data to get each order line's price and customer's birth date.
lines = orders.merge(products, on="product_id", validate="many_to_one")
lines = lines.merge(customers, on="customer_id", validate="many_to_one")
lines["line_revenue_cents"] = lines["quantity_product"] * lines["price"]

# Calculate completed age, including whether the birthday has passed.
birthdays_passed_at_order = (
    (lines["date"].dt.month > lines["birth_date"].dt.month)
    | (
        (lines["date"].dt.month == lines["birth_date"].dt.month)
        & (lines["date"].dt.day >= lines["birth_date"].dt.day)
    )
)
lines["age_at_order"] = (
    lines["date"].dt.year
    - lines["birth_date"].dt.year
    - (~birthdays_passed_at_order).astype(int)
)

# Filters for each question, expressed separately from the SQL queries.
case_filters = {
    "C01": {"period": ("2026-03-01", "2026-06-01")},
    "C02": {"age_reference": "2026-09-25"},
    "C03": {"period": ("2026-01-01", "2026-07-01"), "products": ["a", "b", "c"]},
    "C04": {"period": ("2026-01-01", "2026-10-01")},
    "C05": {"period": ("2026-06-01", "2026-06-02")},
    "C06": {"period": ("2025-01-01", "2026-01-01")},
    "C07": {"period": ("2026-05-31", "2026-06-01")},
    "C08": {"age_reference": "2026-09-24"},
    "C09": {"age_reference": "2026-09-26"},
    "C10": {"age_at_order": True},
    "C11": {"age_reference": "2026-12-01"},
    "C12": {"products": ["a", "b", "c"]},
    "C13": {"period": ("2026-04-01", "2026-05-01"), "products": ["a", "c"]},
    "C14": {"period": ("2026-10-01", "2026-12-01"), "products": ["a", "b", "c"]},
    "C15": {"period": ("2026-03-01", "2026-06-01"), "age_reference": "2026-09-25"},
    "C16": {"age_reference": "2026-09-25", "products": ["a", "b", "c"]},
    "C17": {"period": ("2026-01-01", "2026-07-01"), "age_reference": "2026-09-25", "products": ["a", "b", "c"]},
    "C18": {"period": ("2026-05-01", "2026-06-01"), "age_at_order": True},
    "C19": {"period": ("2026-09-01", "2026-10-01"), "age_reference": "2026-09-25", "products": ["d"]},
    "C20": {"period": ("2026-06-01", "2026-09-01")},
}

expected_ids = [f"C{number:02d}" for number in range(1, 21)]
assert sql_reference["case_id"].tolist() == expected_ids
assert set(case_filters) == set(expected_ids)
assert sql_reference["revenue_cents"].notna().all()

csv_results = []
for case_id, filters in case_filters.items():
    selected = pd.Series(True, index=lines.index)

    if "period" in filters:
        start, end = filters["period"]
        selected &= (lines["date"] >= pd.Timestamp(start)) & (lines["date"] < pd.Timestamp(end))

    if "age_reference" in filters:
        reference_date = pd.Timestamp(filters["age_reference"])
        birthday_passed = (
            (lines["birth_date"].dt.month < reference_date.month)
            | (
                (lines["birth_date"].dt.month == reference_date.month)
                & (lines["birth_date"].dt.day <= reference_date.day)
            )
        )
        age_on_reference = (
            reference_date.year
            - lines["birth_date"].dt.year
            - (~birthday_passed).astype(int)
        )
        selected &= age_on_reference == 25

    if filters.get("age_at_order"):
        selected &= lines["age_at_order"] == 25

    if "products" in filters:
        selected &= lines["product_id"].isin(filters["products"])

    csv_results.append({
        "case_id": case_id,
        "csv_revenue_cents": int(lines.loc[selected, "line_revenue_cents"].sum()),
    })

csv_results = pd.DataFrame(csv_results)
comparison = sql_reference[["case_id", "revenue_cents"]].merge(
    csv_results, on="case_id", how="outer", validate="one_to_one", indicator=True
)
assert (comparison["_merge"] == "both").all(), "Des identifiants de question manquent dans un des fichiers."
comparison = comparison.rename(columns={"revenue_cents": "sql_revenue_cents"})
comparison["status"] = comparison.apply(
    lambda row: "OK" if row["sql_revenue_cents"] == row["csv_revenue_cents"] else "ÉCART",
    axis=1,
)

print("Comparaison : valeurs SQL de reference-td1.csv vs nouveau calcul pandas sur les CSV")
display(comparison[["case_id", "sql_revenue_cents", "csv_revenue_cents", "status"]])
assert comparison["status"].eq("OK").all(), "Au moins un résultat SQL diffère du calcul indépendant pandas."
print(f"Les {len(comparison)} résultats SQL correspondent au calcul indépendant des CSV.")

# Run one query

In [ ]:
import re
import sqlite3
from pathlib import Path

def run_one_query(case_id):
    """Run one reference query by its ID and return revenue_cents."""
    sql_path = Path.cwd() / "reference-td1.sql"
    database_path = Path.cwd() / "data" / "boutique.sqlite"
    matching_queries = []
    for block in sql_path.read_text(encoding="utf-8").split(";"):
        case_match = re.search(r"--\s*(C\d{2})\s*\|", block)
        if case_match and case_match.group(1) == case_id:
            query = re.sub(r"(?m)^\s*--.*$", "", block).strip()
            if query:
                matching_queries.append(query)
    if not matching_queries:
        raise ValueError(f"Question inconnue : {case_id}")
    if len(matching_queries) > 1:
        raise ValueError(f"Plusieurs requêtes trouvées pour : {case_id}")

    database_uri = database_path.resolve().as_uri() + "?mode=ro"
    with sqlite3.connect(database_uri, uri=True) as connection:
        cursor = connection.execute(matching_queries[0])
        rows = cursor.fetchall()

    if [column[0] for column in cursor.description] != ["revenue_cents"]:
        raise ValueError("La requête doit retourner une colonne revenue_cents.")
    if len(rows) != 1 or len(rows[0]) != 1:
        raise ValueError("La requête doit retourner exactement une ligne et une colonne.")

    return rows[0][0]

# Change C01 to another question ID, from C01 to C20.
case_id = "C01"
revenue_cents = run_one_query(case_id)
print(f"{case_id}: {revenue_cents} centimes d'euro hors TVA")